## Reproduction maintenance (2026-11-25)
Historical cell outputs are retained from the original execution, not regenerated by this maintenance commit. Input resolution and dependency pinning were updated; new outputs are written to a fresh temporary directory outside committed evidence. Restart the kernel and start from the repository root or assignment folder before reproducing. Assignment 4 requires explicit confirmation and must never be used for tuning.


# INAI 6986 — Assignment 3 / Milestone 1
## Data & Model Selection

**Project:** Lightweight On-Premises AI-Assisted IT Work Triage for Generalist State Agency IT Teams

This notebook implements the Milestone 1 modeling plan from the source-of-truth Proposal, PM Plan/Log 1, and Log 2.

### Protected-test rule
This notebook uses **only Train (120) and Validation (40)**.  
The frozen 40-case Test set is intentionally excluded from this notebook and must not be used for baseline-rule tuning, feature selection, candidate selection, or threshold tuning.

### Models
1. Fixed keyword/risk-rule baseline
2. TF-IDF + Logistic Regression
3. TF-IDF + Linear SVM
4. TF-IDF + Naive Bayes

### Primary metric
**Recall for Escalation Recommended (label = 1)**

Secondary evidence: precision, F1, false-negative count, accuracy, escalation rate, and runtime.


## 1. Setup

In Google Colab, run the next cell. If the development workbook is not already present, Colab will ask you to upload:

`Williams_INAI6986_Development_Train_Validation_v1_0.xlsx`

Do **not** upload the frozen Test file for Assignment 3.


In [12]:
!pip -q install pandas scikit-learn==1.6.1 joblib openpyxl


In [13]:
import os
import re
import time
import json
import hashlib
import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    recall_score, precision_score, f1_score, accuracy_score,
    confusion_matrix, classification_report
)
from sklearn.metrics.pairwise import cosine_similarity

from pathlib import Path
import tempfile

def locate_input(relative_path):
    for root in (Path.cwd(), *Path.cwd().parents):
        candidate = root / relative_path
        if candidate.is_file():
            return str(candidate.resolve())
    return Path(relative_path).name

DATA_FILE = locate_input("Data/Williams_INAI6986_Development_Train_Validation_v1_0.xlsx")
SHEET_NAME = "Development_Data"

if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print(f"Upload {DATA_FILE}")
        uploaded = files.upload()
    except Exception:
        pass

assert os.path.exists(DATA_FILE), f"{DATA_FILE} was not found."

DATA_FILE = str(Path(DATA_FILE).resolve())
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="it_triage_assignment3_"))
print("Reproduction output directory:", OUTPUT_DIR)
os.chdir(OUTPUT_DIR)

df = pd.read_excel(DATA_FILE, sheet_name=SHEET_NAME)

required = {"scenario_id","domain","scenario_text","escalation_label","split","source_batch"}
missing = required - set(df.columns)
assert not missing, f"Missing required columns: {missing}"

assert set(df["split"].dropna().unique()).issubset({"Train","Validation"}),     "Protected Test rows detected. Stop and use the development-only workbook."

assert len(df) == 160, f"Expected 160 development rows; found {len(df)}."

print("Dataset loaded successfully.")
print(df["split"].value_counts())
print("\nLabel distribution:")
print(df["escalation_label"].value_counts().sort_index())
print("\nDomain x split:")
print(pd.crosstab(df["domain"], df["split"]))


Dataset loaded successfully.
split
Train         120
Validation     40
Name: count, dtype: int64

Label distribution:
escalation_label
0    80
1    80
Name: count, dtype: int64

Domain x split:
split                   Train  Validation
domain                                   
Backup/DR                  30          10
Infrastructure             30          10
Procurement/Vendor/RFP     30          10
Security/Policy            30          10


## 2. Define Train and Validation

Only `scenario_text` is used as the classifier input.  
The target is `escalation_label`.

Metadata, rationale, reviewer fields, notes, difficulty, signal type, and alignment fields are not model inputs.


In [14]:
train_df = df[df["split"] == "Train"].copy()
val_df = df[df["split"] == "Validation"].copy()

X_train = train_df["scenario_text"].astype(str)
y_train = train_df["escalation_label"].astype(int)

X_val = val_df["scenario_text"].astype(str)
y_val = val_df["escalation_label"].astype(int)

assert len(train_df) == 120
assert len(val_df) == 40
assert y_train.value_counts().to_dict() == {0: 60, 1: 60}
assert y_val.value_counts().to_dict() == {0: 20, 1: 20}

print("Train:", len(train_df), y_train.value_counts().to_dict())
print("Validation:", len(val_df), y_val.value_counts().to_dict())


Train: 120 {0: 60, 1: 60}
Validation: 40 {0: 20, 1: 20}


## 3. Fixed Keyword/Risk-Rule Baseline

This baseline is intentionally simple and deterministic. It represents a rule-based screening approach rather than a learned model.

**Important:** Freeze this list before looking at candidate-model validation performance. If you edit the list after examining the ML results, document that as a new baseline version instead of silently changing it.


In [15]:
BASELINE_RISK_PATTERNS = [
    r"\bunsupported\b",
    r"\bend[- ]of[- ]support\b",
    r"\bdisable\b.*\bmfa\b",
    r"\bexempt\b.*\bmfa\b",
    r"\bpermanent\b.*\badmin",
    r"\bprivileged\b.*\baccess\b",
    r"\bpublic internet\b",
    r"\bexpose\b.*\bpublic\b",
    r"\bremove\b.*\bencryption\b",
    r"\bdisable\b.*\bencryption\b",
    r"\bshorten\b.*\bretention\b",
    r"\breduce\b.*\bretention\b",
    r"\bchange\b.*\brpo\b",
    r"\bchange\b.*\brto\b",
    r"\brecovery point objective\b",
    r"\brecovery time objective\b",
    r"\bpolicy exception\b",
    r"\bpolicy waiver\b",
    r"\bsecurity exception\b",
    r"\bbypass\b.*\bpolicy\b",
    r"\bnot included in the executed contract\b",
    r"\boutside\b.*\bcontract\b",
    r"\badditional implementation work\b",
    r"\bdiagnostic copies\b.*\bagency data\b",
    r"\bstop backing up\b",
    r"\bturn off centralized logging\b",
    r"\bexclude\b.*\bendpoint protection\b",
]

baseline_regex = re.compile("|".join(BASELINE_RISK_PATTERNS), flags=re.IGNORECASE)

def baseline_predict(texts):
    return np.array([1 if baseline_regex.search(str(t)) else 0 for t in texts], dtype=int)

print(f"Baseline version 1 contains {len(BASELINE_RISK_PATTERNS)} fixed risk patterns.")


Baseline version 1 contains 27 fixed risk patterns.


## 4. Evaluation Helper

All systems are evaluated on the same 40-case Validation split.


In [16]:
def evaluate_predictions(name, y_true, y_pred, runtime_sec):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "System": name,
        "Recall_Escalation": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Precision_Escalation": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1_Escalation": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "False_Negatives": int(fn),
        "False_Positives": int(fp),
        "Escalation_Rate": float(np.mean(y_pred)),
        "Runtime_sec": float(runtime_sec),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }

results = []

start = time.perf_counter()
baseline_val_pred = baseline_predict(X_val)
baseline_runtime = time.perf_counter() - start
results.append(evaluate_predictions(
    "Keyword/Risk Baseline", y_val, baseline_val_pred, baseline_runtime
))

pd.DataFrame(results)


,System,Recall_Escalation,Precision_Escalation,F1_Escalation,Accuracy,False_Negatives,False_Positives,Escalation_Rate,Runtime_sec,TN,FP,FN,TP
0,Keyword/Risk Baseline,0.1,1.0,0.181818,0.55,18,0,0.05,0.00153,20,0,18,2


## 5. Candidate Models

The same TF-IDF representation is used across candidates for a fair comparison.  
No Test data is used.


In [17]:
candidates = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1,2))),
        ("model", LogisticRegression(max_iter=2000, random_state=6986)),
    ]),
    "Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1,2))),
        ("model", LinearSVC(random_state=6986)),
    ]),
    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1,2))),
        ("model", MultinomialNB()),
    ]),
}

fitted = {}
val_predictions = {}

for name, pipe in candidates.items():
    start = time.perf_counter()
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)
    runtime = time.perf_counter() - start

    fitted[name] = pipe
    val_predictions[name] = pred
    results.append(evaluate_predictions(name, y_val, pred, runtime))

comparison = pd.DataFrame(results)
comparison = comparison.sort_values(
    by=["Recall_Escalation","F1_Escalation","Precision_Escalation","Runtime_sec"],
    ascending=[False,False,False,True]
).reset_index(drop=True)

display(comparison[[
    "System","Recall_Escalation","Precision_Escalation","F1_Escalation",
    "False_Negatives","False_Positives","Accuracy","Escalation_Rate","Runtime_sec"
]])


,System,Recall_Escalation,Precision_Escalation,F1_Escalation,False_Negatives,False_Positives,Accuracy,Escalation_Rate,Runtime_sec
0,Naive Bayes,0.90,1.000000,0.947368,2,0,0.950,0.450,0.022920
1,Linear SVM,0.85,0.944444,0.894737,3,1,0.900,0.450,0.050789
2,Logistic Regression,0.85,0.894737,0.871795,3,2,0.875,0.475,0.056674
3,Keyword/Risk Baseline,0.10,1.000000,0.181818,18,0,0.550,0.050,0.001530


## 6. Validation Confusion Matrices and False Negatives

For this business problem, false negatives deserve special attention because they represent work that should be escalated but is predicted as routine.


In [18]:
all_predictions = {"Keyword/Risk Baseline": baseline_val_pred, **val_predictions}

for name, pred in all_predictions.items():
    print("\n" + "="*80)
    print(name)
    print("Confusion matrix [rows=true, cols=predicted]:")
    print(confusion_matrix(y_val, pred, labels=[0,1]))
    print(classification_report(y_val, pred, digits=3, zero_division=0))

    tmp = val_df[["scenario_id","domain","scenario_text","escalation_label"]].copy()
    tmp["prediction"] = pred
    fn = tmp[(tmp["escalation_label"] == 1) & (tmp["prediction"] == 0)]
    print(f"False negatives: {len(fn)}")
    if len(fn):
        display(fn[["scenario_id","domain","scenario_text"]])



Keyword/Risk Baseline
Confusion matrix [rows=true, cols=predicted]:
[[20  0]
 [18  2]]
              precision    recall  f1-score   support

           0      0.526     1.000     0.690        20
           1      1.000     0.100     0.182        20

    accuracy                          0.550        40
   macro avg      0.763     0.550     0.436        40
weighted avg      0.763     0.550     0.436        40

False negatives: 18


,scenario_id,domain,scenario_text
4,INF-007,Infrastructure,Install an operating-system release that is ou...
12,SEC-007,Security/Policy,Disable multi-factor authentication for an app...
28,PRC-007,Procurement/Vendor/RFP,Accept a hosted vendor solution that stores ag...
36,INF-017,Infrastructure,Disable application health checks in the load ...
44,SEC-017,Security/Policy,Ignore repeated endpoint alerts from one execu...
52,BDR-016,Backup/DR,Promise business leadership a one-hour recover...
68,INF-027,Infrastructure,Combine production and test workloads on the s...
76,SEC-027,Security/Policy,Allow a system administrator to use a personal...
84,BDR-027,Backup/DR,Change the recovery plan to restore only the a...
93,PRC-027,Procurement/Vendor/RFP,Waive the requirement for audit-log export bec...



Logistic Regression
Confusion matrix [rows=true, cols=predicted]:
[[18  2]
 [ 3 17]]
              precision    recall  f1-score   support

           0      0.857     0.900     0.878        20
           1      0.895     0.850     0.872        20

    accuracy                          0.875        40
   macro avg      0.876     0.875     0.875        40
weighted avg      0.876     0.875     0.875        40

False negatives: 3


,scenario_id,domain,scenario_text
36,INF-017,Infrastructure,Disable application health checks in the load ...
84,BDR-027,Backup/DR,Change the recovery plan to restore only the a...
107,INF-046,Infrastructure,A replacement network device is functionally s...



Linear SVM
Confusion matrix [rows=true, cols=predicted]:
[[19  1]
 [ 3 17]]
              precision    recall  f1-score   support

           0      0.864     0.950     0.905        20
           1      0.944     0.850     0.895        20

    accuracy                          0.900        40
   macro avg      0.904     0.900     0.900        40
weighted avg      0.904     0.900     0.900        40

False negatives: 3


,scenario_id,domain,scenario_text
36,INF-017,Infrastructure,Disable application health checks in the load ...
84,BDR-027,Backup/DR,Change the recovery plan to restore only the a...
107,INF-046,Infrastructure,A replacement network device is functionally s...



Naive Bayes
Confusion matrix [rows=true, cols=predicted]:
[[20  0]
 [ 2 18]]
              precision    recall  f1-score   support

           0      0.909     1.000     0.952        20
           1      1.000     0.900     0.947        20

    accuracy                          0.950        40
   macro avg      0.955     0.950     0.950        40
weighted avg      0.955     0.950     0.950        40

False negatives: 2


,scenario_id,domain,scenario_text
36,INF-017,Infrastructure,Disable application health checks in the load ...
107,INF-046,Infrastructure,A replacement network device is functionally s...


## 7. Selection Rule

The project committed to **escalation recall first**, then precision/F1, interpretability, CPU runtime, and maintainability.

The next cell creates an evidence-based ranking. The final written justification should still be reviewed by you rather than relying on the ranking mechanically.


In [19]:
candidate_only = comparison[comparison["System"] != "Keyword/Risk Baseline"].copy()

selected_name = candidate_only.iloc[0]["System"]
selected_model = fitted[selected_name]

print("Top-ranked candidate under the committed validation rule:")
print(selected_name)

display(candidate_only[[
    "System","Recall_Escalation","Precision_Escalation","F1_Escalation",
    "False_Negatives","False_Positives","Runtime_sec"
]])


Top-ranked candidate under the committed validation rule:
Naive Bayes


,System,Recall_Escalation,Precision_Escalation,F1_Escalation,False_Negatives,False_Positives,Runtime_sec
0,Naive Bayes,0.90,1.000000,0.947368,2,0,0.022920
1,Linear SVM,0.85,0.944444,0.894737,3,1,0.050789
2,Logistic Regression,0.85,0.894737,0.871795,3,2,0.056674


## 8. Interpretability Check

For the selected model, inspect terms most associated with escalation and routine handling.  
The purpose is to ask whether the learned signals make business sense and whether obvious label leakage is present.


In [20]:
vectorizer = selected_model.named_steps["tfidf"]
model = selected_model.named_steps["model"]
terms = np.array(vectorizer.get_feature_names_out())

if hasattr(model, "coef_"):
    weights = model.coef_[0]
elif hasattr(model, "feature_log_prob_"):
    weights = model.feature_log_prob_[1] - model.feature_log_prob_[0]
else:
    raise RuntimeError("Selected model does not expose interpretable feature weights.")

top_n = 20
top_escalate_idx = np.argsort(weights)[-top_n:][::-1]
top_routine_idx = np.argsort(weights)[:top_n]

interpretability = pd.DataFrame({
    "Top_Escalation_Term": terms[top_escalate_idx],
    "Escalation_Weight": weights[top_escalate_idx],
    "Top_Routine_Term": terms[top_routine_idx],
    "Routine_Weight": weights[top_routine_idx],
})

display(interpretability)


,Top_Escalation_Term,Escalation_Weight,Top_Routine_Term,Routine_Weight
0,production,0.955939,the approved,-0.809175
1,though,0.820940,verify,-0.732586
2,even though,0.820940,already,-0.724738
3,even,0.820940,and,-0.720683
4,not,0.783228,supported,-0.710360
5,though the,0.702549,documented,-0.694313
6,because,0.698008,using the,-0.665025
7,reduce,0.605280,approved,-0.656234
8,cannot,0.597749,to the,-0.626224
9,or,0.588979,the documented,-0.618922


## 9. Leakage / Duplicate Checks

These checks use Train and Validation only.

They verify:
- no exact scenario-text duplicate crosses Train/Validation,
- highly similar cross-split pairs are identified for human inspection,
- obvious label-language is not embedded directly in scenario text.


In [21]:
# Exact duplicates across Train and Validation
train_text_set = set(X_train.str.strip().str.lower())
val_text_set = set(X_val.str.strip().str.lower())
exact_cross_split = train_text_set.intersection(val_text_set)

print("Exact Train/Validation duplicate texts:", len(exact_cross_split))

# Near-duplicate check using character n-grams
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), lowercase=True)
combined = pd.concat([X_train, X_val], ignore_index=True)
char_matrix = char_vec.fit_transform(combined)

train_mat = char_matrix[:len(X_train)]
val_mat = char_matrix[len(X_train):]

sim = cosine_similarity(val_mat, train_mat)
pairs = []
for i in range(sim.shape[0]):
    j = int(np.argmax(sim[i]))
    score = float(sim[i,j])
    pairs.append({
        "validation_id": val_df.iloc[i]["scenario_id"],
        "train_id": train_df.iloc[j]["scenario_id"],
        "similarity": score,
        "validation_text": val_df.iloc[i]["scenario_text"],
        "train_text": train_df.iloc[j]["scenario_text"],
    })

near = pd.DataFrame(pairs).sort_values("similarity", ascending=False)
print("Top cross-split similarities:")
display(near.head(10)[["validation_id","train_id","similarity"]])

# Explicit label-language scan
leak_phrases = [
    "escalation recommended",
    "generalist handling appropriate",
    "reviewer label",
    "project label",
    "adjudicated label",
]
for phrase in leak_phrases:
    count = df["scenario_text"].str.contains(phrase, case=False, na=False).sum()
    print(f"{phrase!r}: {count}")


Exact Train/Validation duplicate texts: 0
Top cross-split similarities:


,validation_id,train_id,similarity
8,INF-012,INF-022,0.561374
5,BDR-007,BDR-025,0.443387
4,BDR-002,BDR-012,0.411357
36,PRC-033,PRC-040,0.403820
6,PRC-002,PRC-013,0.374865
21,BDR-027,BDR-049,0.369968
27,INF-048,INF-042,0.348875
18,SEC-023,SEC-035,0.343328
32,BDR-036,BDR-012,0.339664
10,SEC-012,SEC-022,0.332469


'escalation recommended': 0
'generalist handling appropriate': 0
'reviewer label': 0
'project label': 0
'adjudicated label': 0


## 10. Save Milestone 1 Evidence

This cell saves:
- validation comparison table,
- validation predictions,
- interpretability terms,
- selected model pipeline,
- SHA-256 hash of the saved model,
- a compact run-log JSON.

The saved pipeline contains both TF-IDF and the selected classifier.


In [22]:
safe_name = selected_name.lower().replace(" ", "_").replace("/", "_")

comparison_file = "assignment3_validation_comparison.csv"
predictions_file = "assignment3_validation_predictions.csv"
terms_file = "assignment3_selected_model_top_terms.csv"
model_file = f"assignment3_selected_model_{safe_name}.joblib"
runlog_file = "assignment3_run_log.json"

comparison.to_csv(comparison_file, index=False)

pred_out = val_df[["scenario_id","domain","scenario_text","escalation_label"]].copy()
pred_out["baseline_prediction"] = baseline_val_pred
for name, pred in val_predictions.items():
    pred_out[name.replace(" ","_").lower()] = pred
pred_out.to_csv(predictions_file, index=False)

interpretability.to_csv(terms_file, index=False)

joblib.dump(selected_model, model_file)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

model_hash = sha256_file(model_file)

run_log = {
    "assignment": "INAI 6986 Assignment 3 / Milestone 1",
    "development_file": DATA_FILE,
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "test_rows_used": 0,
    "primary_metric": "Recall for Escalation Recommended",
    "selection_rule": "Recall first, then F1/precision, interpretability, runtime, maintainability",
    "selected_model": selected_name,
    "selected_model_file": model_file,
    "selected_model_sha256": model_hash,
    "baseline_pattern_count": len(BASELINE_RISK_PATTERNS),
    "comparison": comparison.to_dict(orient="records"),
}

with open(runlog_file, "w") as f:
    json.dump(run_log, f, indent=2)

print("Selected model:", selected_name)
print("Model file:", model_file)
print("Model SHA-256:", model_hash)
print("\nSaved evidence files:")
for f in [comparison_file, predictions_file, terms_file, model_file, runlog_file]:
    print(" -", f)


Selected model: Naive Bayes
Model file: assignment3_selected_model_naive_bayes.joblib
Model SHA-256: e7e440cacfc5926e4eab482bc9baad188eebedf2dbab22b21b3d49fcffac4239

Saved evidence files:
 - assignment3_validation_comparison.csv
 - assignment3_validation_predictions.csv
 - assignment3_selected_model_top_terms.csv
 - assignment3_selected_model_naive_bayes.joblib
 - assignment3_run_log.json


## 11. Milestone 1 Write-Up Checklist

Use the outputs above to complete the official Assignment 3 sections:

1. **Data characteristics** — 200-row master; 120 Train / 40 Validation / 40 protected Test; 100/100 overall target balance; four domains.
2. **Candidates considered** — Logistic Regression, Linear SVM, Naive Bayes.
3. **Baseline** — fixed keyword/risk-rule classifier and Validation recall.
4. **Evidence** — same-split comparison table.
5. **Selection & justification** — selected model plus business/operational reasons.
6. **Interpretability check** — top terms + duplicate/leakage findings.
7. **What would change my mind** — evidence that would favor another candidate.
8. **Repository** — notebook and generated evidence files.

### Stop point
Do **not** evaluate the frozen 40-case Test set in this notebook.  
Assignment 4 / Milestone 2 begins only after the model and any decision threshold are frozen.
